In [1]:
from gliner import GLiNER
from gliner import GLiNER
import os
os.getcwd()

import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
labels = ["event", "place", "space"]
labels2 = ["event", "location", "building"]
labels3 = ["event", "location", "space"]

def extract_gliner(input_text, labels=labels2, confidence_threshold=0.3):

    # initiating dictionary to store extracted entities
    entities_dict = defaultdict(list)

    # extracting entities using GLINER
    entities = model.predict_entities(input_text, labels, threshold=confidence_threshold)
    
    # grouping entities by label and adding to dictionary
    for entity in entities:
        entities_dict[entity['label']].append(entity['text'])

    for label in labels:
        if label in entities_dict and entities_dict[label]:
            # Remove duplicates while preserving case-insensitive uniqueness
            # Keep the first occurrence of each unique entity (case-insensitive)
            unique_entities = []
            seen_lower = set()
            
            for entity in entities_dict[label]:
                entity_lower = entity.lower()
                if entity_lower not in seen_lower:
                    unique_entities.append(entity)
                    seen_lower.add(entity_lower)
            
            # Join the list of unique entities into a single, comma-separated string
            formatted_list = ", ".join(unique_entities)
            print(f"{label}: [{formatted_list}]")
        else:
            print(f"{label}: No entities found")

In [5]:
text = """
STALYBRIDGE.—A public meeting was held in the People’s School here on Monday evening last, when the National Petition was read and adopted; after which, Mr. James Leach, of Manchester, delivered an address, exposing the fallacies of the Corn Law repealers. A Corn Law lecture had been previously delivered in the town, by a Mr. Spencer, to about half a dozen of the middle classes; the Chartists, however, upset his meeting.  WOOLWICH.—STRIKE OF THE MASONS.—A public meeting of the inhabitants of Woolwich was held on Thursday evening, Oct. 28th, in the theatre of that town, for the purpose of laying before the inhabitants every particular connected with the strike of the masons at the New Houses of Parliament, Nelson’s Monument, and Woolwich Dock Yard, also, to take into consideration the conduct of a portion of the metropolitan press. The meeting was called for seven o’clock, and long before that hour, the theatre was thronged in every part, the boxes being filled with well-dressed females. Mr. Maddox was called to the chair, and the meeting was addressed by Mr. Davies, Mr. Carter, Mr. Wood, Mr. Parker, Mr. Walton, Dr. McDouall, Captain Aokerley, and others. The meeting consisted of about a thousand persons. We are obliged to the kindness of a friend for furnishing us with a long report of this meeting, a favour which would have been greatly enhanced had it reached us before Thursday morning last; just one week after the meeting had been held, and too late to be made use of at length for the Star.
"""
extract_gliner(text)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


event: [National Petition, Corn Law lecture, STRIKE OF THE MASONS, public meeting, strike]
location: [STALYBRIDGE, Manchester, WOOLWICH, theatre]
building: [People’s School, New Houses of Parliament, Nelson’s Monument]


In [9]:
simple_test = """
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"""

labels = ["event", "location"]
extract_gliner(simple_test, labels=labels, confidence_threshold=0.39)

event: [Wednesday evening]
location: [Charlestown meeting room]


In [31]:
test2 = """
EMERR'S BRIGADE.—A public meeting, in favour of the People's Charter, will be held at the Bricklayers' Arms, Homer-street, New Road, Mary-lebone, on Monday evening next, the 18th inst., at eight o'clock precisely. Messrs. Mantz and Davoé will attend.  MR. SKELTON will deliver a lecture at the Standard of Liberty, Brick-lane, Spitalfields, on Sunday evening next, the 17th inst., at half-past seven precisely.  SOMERS' TOWN LOCALITY.—On Sunday evening next, Mr. Mee will lecture at Mr. Dudbridges, Bricklayers' Arms, Tonbridge-street, New Road.  MR. HUNNIBALL, of Stafford, will deliver a lecture on Sunday, the 17th inst., at the Golden Lion, Dean-street, Soho, on the causes of the Revolutions of Greece and Rome.  LONDON DISTRICT COUNCIL.—This Council will meet at the City of London Political and Scientific Institution, Turnagain Lane, on Sunday afternoon next, the 17th inst., at three o'clock precisely.  CAMBERWELL.—A public meeting will be held at the Cook Tavern, Camberwell Green, on Tuesday next, the 19th inst., at eight precisely.  HAMMERSMITH, NOTTINGHILL, AND THEIR VICINITIES.—The Chartists and their friends of the above places are most respectfully requested to attend a meeting at the Black Bull Inn, Hammersmith Road, on Tuesday evening next, the 19th inst., at eight precisely, on business of great importance."""

labels = ["event", "location"]
extract_gliner(test2, labels=labels, confidence_threshold=0.3)

event: [Revolutions of Greece and Rome, public meeting]
location: [Bricklayers' Arms, Homer-street, New Road, Mary-lebone, Standard of Liberty, Brick-lane, Spitalfields, SOMERS' TOWN LOCALITY, Tonbridge-street, Stafford, Golden Lion, Dean-street, Soho, City of London Political and Scientific Institution, Turnagain Lane, CAMBERWELL, Cook Tavern, Camberwell Green, HAMMERSMITH, NOTTINGHILL, Black Bull Inn, Hammersmith Road]
